# Ramadhan et al. (2025) — Daily PharmaSales Final-Pipeline Alignment

Notebook ini eksplorasi preprocessing berdasarkan penelitian sebelumnya. Fokusnya menjalankan model Ramadhan daily PharmaSales (LR-XGBoost) dengan preprocessing pipeline dari penelitian kami, bukan dari paper referensi (paper referensi belum dikirim).

**Pipeline:**
- Dataset: `data/raw/pharma-sales/salesdaily.csv`
- Granularitas: daily
- Unit observasi: 8 kategori ATC
- Lag selection: ACF per kategori, `max_lag=26`
- Feature: `lag_1 ... lag_n` + `rolling_mean_n` dengan `shift(1)`
- Split: chronological `80% train / 20% test`
- Final fit: train only
- Target scale: original daily sales scale, tanpa target scaling
- Metric: Train MSE, Test MSE, Train RMSE, dan Test RMSE per ATC


In [9]:
import os
import sys

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from statsmodels.tsa.stattools import acf

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)


# Exploration Based on Our Preprocessing

Eksplorasi menggunakan preprocessing pipeline dari penelitian kami sebelumnya.


## Preprocessing

Pipeline preprocessing meliputi load data harian, ACF lag selection per kategori, feature engineering (lag features + rolling mean), dan chronological split.


In [10]:
DATA_PATH = "../data/raw/pharma-sales/salesdaily.csv"
CATEGORIES = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
MAX_LAG = 26
RANDOM_STATE = 42

data = pd.read_csv(DATA_PATH)
data["datum"] = pd.to_datetime(data["datum"])
data = data.sort_values("datum").reset_index(drop=True)

data[["datum"] + CATEGORIES].head()

,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0
3,2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0
4,2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0


In [11]:
max_lags = {}

for category in CATEGORIES:
    acf_values = acf(data[category], nlags=MAX_LAG)
    max_lag = np.argmax(acf_values[1:]) + 1
    max_lags[category] = max_lag

max_lags

{'M01AB': np.int64(14),
 'M01AE': np.int64(14),
 'N02BA': np.int64(7),
 'N02BE': np.int64(7),
 'N05B': np.int64(1),
 'N05C': np.int64(4),
 'R03': np.int64(15),
 'R06': np.int64(3)}

In [12]:
transformed_categories = []

for category in CATEGORIES:
    dfg = data[["datum", category]].rename(columns={"datum": "ds", category: "y"}).copy()
    lag_selected = max_lags[category]

    for lag in range(1, lag_selected + 1):
        dfg[f"lag_{lag}"] = dfg["y"].shift(lag)

    dfg[f"rolling_mean_{lag_selected}"] = dfg["y"].shift(1).rolling(window=lag_selected).mean()
    dfg = dfg.dropna().reset_index(drop=True)

    train_size = int(len(dfg) * 0.80)

    dfg_train = dfg.iloc[:train_size]
    dfg_test = dfg.iloc[train_size:]

    feature_columns = [column for column in dfg.columns if column not in ["ds", "y"]]

    transformed_categories.append({
        "Category": category,
        "n_lags": lag_selected,
        "feature_columns": feature_columns,
        "X_train": dfg_train[feature_columns].to_numpy(),
        "y_train": dfg_train["y"].to_numpy(),
        "X_test": dfg_test[feature_columns].to_numpy(),
        "y_test": dfg_test["y"].to_numpy(),
    })

pd.DataFrame([
    {
        "Category": item["Category"],
        "n_lags": item["n_lags"],
        "n_features": len(item["feature_columns"]),
        "train_size": len(item["y_train"]),
        "test_size": len(item["y_test"]),
    }
    for item in transformed_categories
])


,Category,n_lags,n_features,train_size,test_size
0,M01AB,14,15,1673,419
1,M01AE,14,15,1673,419
2,N02BA,7,8,1679,420
3,N02BE,7,8,1679,420
4,N05B,1,2,1684,421
5,N05C,4,5,1681,421
6,R03,15,16,1672,419
7,R06,3,4,1682,421


### LR-XGBoost

Hybrid ensemble: Linear Regression dan XGBoost dilatih secara independen pada data yang sama (X_train, y_train), kemudian hasil prediksi dirata-rata: `(lr_pred + xgb_pred) / 2`. GridSearchCV dengan TimeSeriesSplit digunakan untuk hyperparameter tuning XGBoost pada data train. Metrik: Train MSE, Test MSE, Train RMSE, dan Test RMSE.


In [13]:
param_grid = {
    "n_estimators": [50, 100, 200, 300, 400, 500],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1, 0.2],
}

results = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    lr_pred_train = lr_model.predict(X_train)
    lr_pred_test = lr_model.predict(X_test)

    tscv = TimeSeriesSplit(n_splits=5)
    xgb_base = xgb.XGBRegressor(
        objective="reg:squarederror",
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.1,
        min_child_weight=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    grid_search = GridSearchCV(
        xgb_base,
        param_grid,
        scoring="neg_mean_squared_error",
        cv=tscv,
    )
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_

    xgb_final = xgb.XGBRegressor(
        objective="reg:squarederror",
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=10.0,
        reg_alpha=0.1,
        min_child_weight=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **best_params,
    )
    xgb_final.fit(X_train, y_train)

    xgb_pred_train = xgb_final.predict(X_train)
    xgb_pred_test = xgb_final.predict(X_test)

    y_pred_train = (lr_pred_train + xgb_pred_train) / 2
    y_pred_test = (lr_pred_test + xgb_pred_test) / 2

    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_rmse = float(np.sqrt(train_mse))
    test_rmse = float(np.sqrt(test_mse))

    results.append({
        "Category": category,
        "n_lags": item["n_lags"],
        "n_estimators": best_params["n_estimators"],
        "learning_rate": best_params["learning_rate"],
        "max_depth": best_params["max_depth"],
        "Train MSE": train_mse,
        "Test MSE": test_mse,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse,
        "split": "chronological 80/20",
        "target_scale": "original daily sales scale; no target scaling",
        "metric_family": "MSE/RMSE",
    })

ramadhan_final_pipeline_results = pd.DataFrame(results)
ramadhan_final_pipeline_results


,Category,n_lags,n_estimators,learning_rate,max_depth,Train MSE,Test MSE,Train RMSE,Test RMSE,split,target_scale,metric_family
0,M01AB,14,50,0.05,3,6.739727,7.880626,2.596098,2.807245,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
1,M01AE,14,50,0.05,3,4.066136,4.647256,2.016466,2.155749,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
2,N02BA,7,50,0.05,3,5.479742,4.299706,2.340885,2.073573,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
3,N02BE,7,50,0.10,3,137.418602,159.065119,11.722568,12.612102,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
4,N05B,1,50,0.05,3,31.452823,18.031724,5.608282,4.246378,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
5,N05C,4,50,0.05,3,1.158611,1.245206,1.076388,1.115888,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
6,R03,15,50,0.05,3,29.552773,65.810795,5.436246,8.112385,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
7,R06,3,50,0.05,3,4.261602,4.954415,2.064365,2.225852,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE


## Final Result

### RMSE


In [14]:
ramadhan_table1_export = pd.DataFrame([{
    "Ref": "Ramadhan et al., 2025",
    "Method": "LR-XGBoost",
    **ramadhan_final_pipeline_results.set_index("Category")["Test RMSE"].reindex(CATEGORIES).to_dict(),
}])

ramadhan_table1_export = ramadhan_table1_export[["Ref", "Method"] + CATEGORIES]
ramadhan_table1_export


,Ref,Method,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,"Ramadhan et al., 2025",LR-XGBoost,2.807245,2.155749,2.073573,12.612102,4.246378,1.115888,8.112385,2.225852


# Summary

## Best Results

LR-XGBoost adalah model hybrid ensemble yang dievaluasi dalam notebook ini. Model menggabungkan Linear Regression dan XGBoost melalui averaging: `(lr_pred + xgb_pred) / 2`.

## Key Insights

- LR-XGBoost ensemble averaging: Linear Regression untuk linear trend + XGBoost untuk non-linear pattern, hasil dirata-rata.
- TimeSeriesSplit (5-fold) digunakan untuk hyperparameter tuning pada data train, menghindari data leakage.
- Preprocessing dengan ACF-based lag selection + rolling mean + chronological split efektif untuk time series forecasting.

## Limitations

- Hanya satu model dievaluasi — perlu perbandingan dengan model lain.
- Hanya 8 kategori ATC — perlu validasi pada kategori tambahan.
- Train/Test metrics bisa berbeda signifikan, menunjukkan potensi overfitting pada beberapa kategori.

## Next Steps

- Evaluasi dengan preprocessing pipeline dari paper referensi (setelah tersedia).
- Tambahkan model baseline lain untuk perbandingan (ARIMA, GRNN, RBFNN, P_NN).
- Eksplorasi external features (musim, hari libur) untuk meningkatkan akurasi.
